# 03 - ML Experiment Scoring

Entrena un modelo con MLflow en Fabric, registra el modelo y genera tablas de scoring.

In [ ]:
# Fabric notebook: 03_features_scoring

from datetime import datetime, timezone
import json
import uuid

import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
from pyspark.sql.functions import col
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

experiment_name = "exp_retail_customer_scoring"
registered_model_name = "retail_customer_repurchase_model"
model_version = "fabric-mlflow-001"
target_days = 60


def yes_no(value):
    text = "" if value is None else str(value).strip().lower()
    return text in {"1", "true", "yes", "si", "sí", "y"}


def minmax(series):
    series = pd.to_numeric(series, errors="coerce").fillna(0)
    span = series.max() - series.min()
    if span == 0:
        return pd.Series([0.5] * len(series), index=series.index)
    return (series - series.min()) / span


def heuristic_score(features):
    monetary = minmax(features["monetary_12m"])
    frequency = minmax(features["frequency_12m"])
    recency = 1 - minmax(features["recency_days"])
    returns = 1 - minmax(features["return_rate"])
    return np.clip(0.35 * monetary + 0.3 * frequency + 0.25 * recency + 0.1 * returns, 0, 1)


def build_features(fact, crm):
    fact = fact[fact["customer_id"].astype(str).str.len() > 0].copy()
    fact["movement_date"] = pd.to_datetime(fact["movement_date"], errors="coerce")
    fact["amount"] = pd.to_numeric(fact["amount"], errors="coerce").fillna(0)
    fact["discount_pct"] = pd.to_numeric(fact["discount_pct"], errors="coerce").fillna(0)
    fact["is_return"] = fact["is_return"].astype(bool)
    max_date = fact["movement_date"].dropna().max()
    observation_date = max_date - pd.Timedelta(days=target_days)
    history = fact[fact["movement_date"] <= observation_date].copy()
    future = fact[
        (fact["movement_date"] > observation_date)
        & (fact["movement_date"] <= observation_date + pd.Timedelta(days=target_days))
        & (~fact["is_return"])
    ]
    grouped = history.groupby("customer_id", dropna=False)
    features = grouped.agg(
        last_purchase_at=("movement_date", "max"),
        frequency_12m=("doc_id_std", "nunique"),
        monetary_12m=("amount", "sum"),
        avg_order_value=("amount", "mean"),
        return_count=("is_return", "sum"),
        category_diversity=("category", "nunique"),
        discount_affinity=("discount_pct", "mean"),
        movement_count=("fact_id", "count"),
    ).reset_index()
    online_count = history[history["channel"].eq("ONLINE")].groupby("customer_id").size().rename("online_count")
    store_count = history[~history["channel"].eq("ONLINE")].groupby("customer_id").size().rename("store_count")
    features = features.merge(online_count, on="customer_id", how="left")
    features = features.merge(store_count, on="customer_id", how="left")
    features[["online_count", "store_count"]] = features[["online_count", "store_count"]].fillna(0)
    features["online_share"] = features["online_count"] / features["movement_count"].replace(0, np.nan)
    features["store_share"] = features["store_count"] / features["movement_count"].replace(0, np.nan)
    features["return_rate"] = features["return_count"] / features["movement_count"].replace(0, np.nan)
    features["recency_days"] = (observation_date - features["last_purchase_at"]).dt.days

    crm = crm.copy()
    crm["customer_id"] = crm["customer_id"].astype(str).str.strip().str.upper()
    crm["fecha_alta_programa"] = pd.to_datetime(crm["fecha_alta_programa"], errors="coerce")
    features = features.merge(
        crm[
            [
                "customer_id",
                "tier_fidelizacion",
                "consentimiento_marketing",
                "fecha_alta_programa",
                "canal_preferido_declarado",
            ]
        ],
        on="customer_id",
        how="left",
    )
    features["marketing_consent"] = features["consentimiento_marketing"].map(yes_no)
    features["tenure_days"] = (observation_date - features["fecha_alta_programa"]).dt.days
    last_channel = history.sort_values("movement_date").groupby("customer_id")["channel"].last()
    features["last_channel"] = features["customer_id"].map(last_channel).fillna(features["canal_preferido_declarado"])
    features["target_repurchase"] = features["customer_id"].isin(set(future["customer_id"])).astype(int)
    features["observation_date"] = observation_date.date().isoformat()

    numeric_defaults = {
        "recency_days": 999,
        "frequency_12m": 0,
        "monetary_12m": 0,
        "avg_order_value": 0,
        "return_rate": 0,
        "discount_affinity": 0,
        "online_share": 0,
        "store_share": 0,
        "category_diversity": 0,
        "tenure_days": 0,
    }
    for column_name, default in numeric_defaults.items():
        features[column_name] = pd.to_numeric(features[column_name], errors="coerce").fillna(default)
    return features.sort_values("customer_id")


def train_and_score(features):
    numeric_columns = [
        "recency_days",
        "frequency_12m",
        "monetary_12m",
        "avg_order_value",
        "return_rate",
        "discount_affinity",
        "online_share",
        "store_share",
        "category_diversity",
        "tenure_days",
    ]
    x = features[numeric_columns].fillna(0)
    y = features["target_repurchase"].astype(int)
    metrics = {"model_name": registered_model_name, "model_version": model_version}

    mlflow.set_experiment(experiment_name)
    with mlflow.start_run(run_name="train_customer_repurchase_model"):
        mlflow.log_param("target_days", target_days)
        mlflow.log_param("features", ",".join(numeric_columns))
        mlflow.log_param("training_rows", len(features))
        mlflow.log_param("positive_rate", float(y.mean()))

        if y.nunique() < 2 or len(features) < 20:
            model = RandomForestClassifier(n_estimators=50, max_depth=3, random_state=42)
            model.fit(x, y)
            scores = heuristic_score(features)
            metrics["training_note"] = "Heuristic scoring used because target has insufficient class diversity."
            mlflow.log_param("fallback", "heuristic")
        else:
            try:
                x_train, x_test, y_train, y_test = train_test_split(
                    x, y, test_size=0.3, random_state=42, stratify=y
                )
                model = make_pipeline(
                    StandardScaler(),
                    LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42),
                )
                model.fit(x_train, y_train)
                scores = model.predict_proba(x)[:, 1]
                test_scores = model.predict_proba(x_test)[:, 1]
                metrics["roc_auc"] = round(float(roc_auc_score(y_test, test_scores)), 4)
                metrics["pr_auc"] = round(float(average_precision_score(y_test, test_scores)), 4)
                mlflow.log_metric("roc_auc", metrics["roc_auc"])
                mlflow.log_metric("pr_auc", metrics["pr_auc"])
                mlflow.log_param("fallback", "none")
            except Exception as exc:
                model = RandomForestClassifier(n_estimators=100, max_depth=4, random_state=42)
                model.fit(x, y)
                scores = model.predict_proba(x)[:, 1]
                metrics["training_note"] = f"Random forest fallback used: {exc}"
                mlflow.log_param("fallback", "random_forest")

        mlflow.log_metric("scored_customers", len(features))
        mlflow.sklearn.log_model(
            sk_model=model,
            artifact_path="model",
            registered_model_name=registered_model_name,
        )
        mssparkutils.fs.mkdirs("Files/generated")
        mssparkutils.fs.put("Files/generated/ml_metrics.json", json.dumps(metrics, indent=2), True)
    return np.round(scores, 6), metrics


fact_pd = spark.table("curated_fact_transactions").toPandas()
crm_pd = spark.table("raw.crm").toPandas()
features_pd = build_features(fact_pd, crm_pd)
score_values, metrics = train_and_score(features_pd)

scores_pd = pd.DataFrame(
    {
        "customer_id": features_pd["customer_id"],
        "score_date": features_pd["observation_date"],
        "model_name": metrics["model_name"],
        "model_version": metrics["model_version"],
        "score": score_values,
    }
)
scores_pd["score_band"] = pd.cut(
    scores_pd["score"],
    bins=[-0.01, 0.35, 0.65, 0.82, 1.0],
    labels=["Low", "Medium", "High", "Risk"],
).astype(str)
scores_pd["recommended_action"] = scores_pd["score_band"].map(
    {
        "High": "Prioritize premium cross-sell.",
        "Medium": "Offer a moderate personalized incentive.",
        "Low": "Use light reactivation or exclude from costly actions.",
        "Risk": "Trigger retention review for high-potential customer.",
    }
)
scores_pd["explanation_json"] = features_pd.apply(
    lambda row: json.dumps(
        {
            "recency_days": row["recency_days"],
            "frequency_12m": row["frequency_12m"],
            "monetary_12m": round(float(row["monetary_12m"]), 2),
            "return_rate": round(float(row["return_rate"]), 4),
        }
    ),
    axis=1,
)
history_pd = scores_pd.copy()
history_pd.insert(0, "score_event_id", [str(uuid.uuid4()) for _ in range(len(history_pd))])
history_pd["created_at"] = datetime.now(timezone.utc).replace(microsecond=0).isoformat()

spark.createDataFrame(features_pd).write.mode("overwrite").format("delta").saveAsTable("customer_features")
spark.createDataFrame(scores_pd.sort_values("score", ascending=False)).write.mode("overwrite").format("delta").saveAsTable(
    "ml_customer_score_current"
)
spark.createDataFrame(history_pd).write.mode("overwrite").format("delta").saveAsTable("ml_customer_score_history")

display(spark.table("ml_customer_score_current").orderBy(col("score").desc()).limit(20))
